# Fluorescence lifetimes of 4 fluorophores at 3 nm distance

In [1]:
import numpy as np

import fluopy.analysis as an
import fluopy.fluorophores as fl
import fluopy.routines as rt
import fluopy.simulation as si
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2

saving_at = (
    r"D:\python_output\Chapter_I\1_13_multi_f_et_adjustments\fluorescence_lifetimes"
)

**Note** The fluorescence lifetimes will be simulated using a continuous excitation regime to accerate the simulation. The analyzed S1 durations will hence represent the time difference of between excitation and emission and not the time difference of previous laser pulse and emission. Also, only S1 durations that end up in fluorescence will be analyzed.

## $<\kappa^2>$ of $\frac{2}{3}$ 

In [2]:
fluorophores = fl.construct_fluorophores(
    name="cy5_dna",
    count=4,
    distance=3,
    shape="square",
)
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
transitions = fluorophore_system.load_transitions(
    bleaching=False,
    summarize=True,
    energy_transfer=True,
    **rt.PARAMS_DSTORM,
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set.finalize()

In [3]:
sim = si.Simulation(transition_set)
sim.run(size=1e8, use_memmap=r"C:\Users\vie43sq\Desktop\Simulations\memmaps\run_2")

Floating point precision error warning:
 The higher limit of smallest increment with a probability of 1.00e-03 is 9.58e-16.
 This was estimated using the highest possible rate which occurs for example in state combination [1, 1, 1, 1].
 Everything drawn below this number will be rounded to zero starting somewhere between 1.00e+00 - 1.00e+01.


In [4]:
analysis = an.Analysis(sim)
fluorescence_lifetimes = analysis.get_emitting_transition_lifetimes("cy5_dna")

In [5]:
fluorescence_lifetimes.mean()

np.float64(1.6341700174301534e-09)

In [38]:
indices = transition_set.combined_state_transitions_df.index[
    transition_set.combined_state_transitions_df["abbreviation"] == "OET_1"
]

In [6]:
1 - 0.042 / (1 + 0.042)

0.9596928982725528

In [ ]:
occurrences = np.isin(sim.transition_series, indices).sum()
occurrences

np.int64(2075931)

: 

In [34]:
sim.state_series

array([0, 1, 3, 6, 7, 9], dtype=int8)

In [25]:
transition_set.transition_df

transition_type  \
Fluorophore                         identity                                           
cy5_dna                             0                      TransitionType.EXCITATION   
                                    1            TransitionType.FLUORESCENT_EMISSION   
                                    2         TransitionType.INTERSYSTEM_CROSSING_ST   
                                    3                   TransitionType.ISOMERIZATION   
                                    4                        TransitionType.ADDUCT_T   
                                    5                        TransitionType.ADDUCT_S   
                                    6                      TransitionType.RAD_ESCAPE   
                                    7                       TransitionType.RAD_RELAX   
                                    8               TransitionType.S1_S0_TRANSITIONS   
                                    9               TransitionType.T1_S0_TRANSITIONS   
                                    10             TransitionType.CIS_S0_TRANSITIONS   
                                    11             TransitionType.OFF_S0_TRANSITIONS   
D: cy5_dna, A: cy5_dna, dist: 3.0   12                     TransitionType.CIS_FRET_1   
                                    13                     TransitionType.CIS_FRET_2   
                                    14                     TransitionType.OFF_FRET_1   
                                    15                     TransitionType.OFF_FRET_2   
                                    16               TransitionType.S_S_ANNIHILATION   
                                    17               TransitionType.S_T_ANNIHILATION   
D: cy5_dna, A: cy5_dna, dist: 4.243 18                     TransitionType.CIS_FRET_1   
                                    19                     TransitionType.CIS_FRET_2   
                                    20                     TransitionType.OFF_FRET_1   
                                    21                     TransitionType.OFF_FRET_2   
                                    22               TransitionType.S_S_ANNIHILATION   
                                    23               TransitionType.S_T_ANNIHILATION   

                                             abbreviation       initial_state  \
Fluorophore                         identity                                    
cy5_dna                             0                 EXC      SingleState.S0   
                                    1                 FLU      SingleState.S1   
                                    2              ISC_ST      SingleState.S1   
                                    3                 ISO      SingleState.S1   
                                    4              PET_TO      SingleState.T1   
                                    5              PET_SO      SingleState.S1   
                                    6              PET_TR      SingleState.T1   
                                    7                 OXI       SingleState.R   
                                    8             S1S0SUM      SingleState.S1   
                                    9             T1S0SUM      SingleState.T1   
                                    10           cisS0SUM     SingleState.cis   
                                    11           OFFS0SUM     SingleState.OFF   
D: cy5_dna, A: cy5_dna, dist: 3.0   12              CET_1  PairedState.S1_Cis   
                                    13              CET_2  PairedState.S1_Cis   
                                    14              OET_1  PairedState.S1_OFF   
                                    15              OET_2  PairedState.S1_OFF   
                                    16                SSA   PairedState.S1_S1   
                                    17                STA   PairedState.S1_T1   
D: cy5_dna, A: cy5_dna, dist: 4.243 18              CET_1  PairedState.S1_Cis   
                                    19              CET_2  PairedState.S1_Cis  

## $<\kappa^2>$ distribution

In [41]:
sim_distr = si.Simulation(transition_set)
kappa_squared = sim_distr.run(
    size=1e8,
    kap_sq_var=True,
    tau_rot=1e-8,
    tau_flu=1e-9,
    dt=1e-11,
    accuracy=20000,
    use_memmap=r"C:\Users\vie43sq\Desktop\Simulations\memmaps\run_1",
)

Floating point precision error warning:
 The higher limit of smallest increment with a probability of 1.00e-03 is 9.58e-16.
 This was estimated using the highest possible rate which occurs for example in state combination [1, 1, 1, 1].
 Everything drawn below this number will be rounded to zero starting somewhere between 1.00e+00 - 1.00e+01.


In [42]:
analysis_distr = an.Analysis(sim_distr)
fluorescence_lifetimes_distr = analysis_distr.get_emitting_transition_lifetimes(
    "cy5_dna"
)

In [ ]:
fluorescence_lifetimes_distr.mean()

np.float64(1.6092783833942344e-09)

: 